# 27. Relaciones con un extremo en una decisión compuesta

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`26-jss-minimo-e3-relaciones-disponibles.ipynb`](26-jss-minimo-e3-relaciones-disponibles.ipynb) ·
[`src/afg/evaluation/reference.py`](../src/afg/evaluation/reference.py) ·
[`src/afg/extraction/openkos/gold_bundle.py`](../src/afg/extraction/openkos/gold_bundle.py) ·
[`tests/test_gold_relations_endpoints.py`](../tests/test_gold_relations_endpoints.py) ·
[ADR 0013](../docs/decisions/0013-adr-composicion-de-la-base-c3.md) ·
[ADR 0021](../docs/decisions/0021-adr-minimo-de-e3-segun-relaciones-disponibles.md)

**Estado: completo. Respalda el ADR 0025 (issue #298).** El notebook cuenta, serie por serie, las
relaciones de oro aceptadas de la Tarea B, comprueba que ninguna tenga como extremo una decisión
`compuesta` y lista las nueve relaciones cuyo extremo se reasignó a una decisión hija.

---

> **Aviso de alcance (ADR 0005).** Las series son series de **evaluación**. El notebook muestra
> solo **identificadores de par y de decisión y conteos**. No muestra ni cita frases de decisión,
> y no contiene ninguna medida del Experimento A ni del B.

# 1. El defecto

La referencia de decisiones (`load_reference`) reemplaza cada decisión `compuesta` por sus hijas
(`d07-1`, `d07-2`, ...): el padre no es una decisión de la referencia. Nueve relaciones aceptadas
de la Tarea B apuntaban sin embargo al identificador del padre. Un enlace así no se puede escribir
en C3 (`build_gold_workspace` no encuentra el extremo y descarta la relación) ni se puede alinear
en el Experimento A (ninguna decisión extraída se alinea con un padre que no está en la
referencia). El criterio adoptado en el ADR 0025 es **una relación, un enlace**: el extremo pasa a
la hija que sostiene el contenido de las decisiones.

In [1]:
import re

import pandas as pd

from afg.evaluation.reference import resolve_reference, resolve_series_file
from afg.experiments.question_bank import evaluation_series
from afg.extraction.openkos.gold_bundle import GOLD_RELATIONS_DIR, load_gold_relations

SERIES = evaluation_series()
print("Series de evaluación:", len(SERIES), "->", ", ".join(SERIES))

Series de evaluación: 11 -> ES2002, ES2008, ES2014, ES2016, IS1003, IS1006, IS1008, IS1009, TS3003, TS3005, TS3011


# 2. Decisiones compuestas y sus hijas

Para cada serie se lee el archivo de referencia (el mismo que resuelve el paquete) y se toman las
filas cuyo estado final es `compuesta`. Sus hijas son las filas `<padre>-<n>`.

In [2]:
STATUS_COLUMNS = ("final_status", "status")


def compuesta_children(series: str) -> dict[str, list[str]]:
    frame = pd.read_csv(resolve_reference(series).path, dtype=str, keep_default_na=False)
    column = next(c for c in STATUS_COLUMNS if c in frame.columns)
    ids = list(frame["decision_id"])
    parents = [i for i, s in zip(ids, frame[column].str.strip(), strict=True) if s == "compuesta"]
    return {p: [i for i in ids if re.fullmatch(re.escape(p) + r"-\d+", i)] for p in parents}


COMPUESTAS = {s: compuesta_children(s) for s in SERIES}
rows = [
    {"serie": s, "compuesta": parent, "hijas": len(kids)}
    for s, parents in COMPUESTAS.items()
    for parent, kids in parents.items()
]
CHILDREN = pd.DataFrame(rows).set_index("serie")
print("Decisiones compuestas en las once series:", len(CHILDREN))
print("Compuestas sin ninguna hija:", int((CHILDREN["hijas"] == 0).sum()))
CHILDREN

Decisiones compuestas en las once series: 18
Compuestas sin ninguna hija: 0


,compuesta,hijas
serie,,
ES2002,ES2002b.d04,4
ES2008,ES2008c.d07,4
ES2014,ES2014b.d04,9
ES2014,ES2014c.d02,2
ES2014,ES2014c.d07,2
ES2014,ES2014d.d01,2
ES2014,ES2014d.d04,2
ES2016,ES2016c.d03,2
IS1008,IS1008b.d02,5


# 3. Relaciones aceptadas por serie y extremos compuestos

Una relación aceptada es la que `load_gold_relations` devuelve: una etiqueta distinta de
`no_relacionada` con la dirección confirmada. La columna «extremo compuesto» cuenta las relaciones
aceptadas cuyo extremo anterior o posterior es una decisión `compuesta`.

In [3]:
rows = []
for s in SERIES:
    relations = load_gold_relations(s)
    parents = set(COMPUESTAS[s])
    offending = [r for r in relations if r.earlier_id in parents or r.later_id in parents]
    rows.append(
        {
            "serie": s,
            "relaciones aceptadas": len(relations),
            "extremo compuesto": len(offending),
        }
    )
ENDPOINTS = pd.DataFrame(rows).set_index("serie")
total = ENDPOINTS.sum()
print("Relaciones aceptadas (suma de las series):", int(total["relaciones aceptadas"]))
print("Con un extremo en una decisión compuesta :", int(total["extremo compuesto"]))
assert int(total["extremo compuesto"]) == 0
ENDPOINTS

Relaciones aceptadas (suma de las series): 37
Con un extremo en una decisión compuesta : 0


,relaciones aceptadas,extremo compuesto
serie,,
ES2002,3,0
ES2008,5,0
ES2014,6,0
ES2016,4,0
IS1003,3,0
IS1006,0,0
IS1008,1,0
IS1009,1,0
TS3003,5,0


# 4. Las nueve relaciones reasignadas

Cada fila reasignada lleva en `razon` (archivo adjudicado) o en `notes` (archivo de un anotador) la
marca «Extremo compuesto reasignado a la hija (X -> Y) por jss, issue #298, ADR 0025.». Aquí se lee
esa marca, de modo que el estado anterior y el posterior salen del propio archivo versionado.

In [4]:
import csv

MARK = re.compile(r"Extremo compuesto reasignado a la hija \((.+?)\) por jss, issue #298, ADR 0025")

rows = []
for s in SERIES:
    path = resolve_series_file(s, GOLD_RELATIONS_DIR, "candidates").path
    with path.open(encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            found = MARK.search(" ".join(row.values()))
            if not found:
                continue
            for change in found.group(1).split(";"):
                before, after = (part.strip() for part in change.split("->"))
                rows.append(
                    {"pair_id": row["pair_id"], "serie": s, "antes": before, "después": after}
                )
REASSIGNED = pd.DataFrame(rows)
print("Extremos reasignados:", len(REASSIGNED), "en", REASSIGNED["pair_id"].nunique(), "relaciones")

Extremos reasignados: 11 en 9 relaciones


El criterio de cada reasignación se resume en la tabla siguiente. Se fija por el contenido de las
decisiones y, cuando las notas de la persona que anotó nombran la hija, se ancla en ellas. Solo
ES2008.p009 es **débil**: no hay ancla textual y se eligió la misma hija que en ES2008.p013 por
consistencia.

In [5]:
CRITERION = {
    ("ES2014.p010", "ES2014b.d04"): "la nota de gm nombra b.d04-3",
    ("ES2014.p011", "ES2014b.d04"): (
        "d.d07 nombra el botón de confirmación; la nota lista -6 a -9 y se conserva una"
    ),
    ("ES2014.p012", "ES2014c.d07"): "la nota de gm nombra c.d07-1",
    ("ES2014.p012", "ES2014d.d01"): "la nota de gm nombra d.d01-1",
    ("ES2014.p014", "ES2014c.d02"): "la nota de gm nombra c.d02-2",
    ("IS1008.p007", "IS1008b.d02"): "la nota de gm nombra b.d02-2",
    ("IS1008.p007", "IS1008c.d02"): "la nota de gm nombra c.d02-5",
    ("ES2016.p001", "ES2016c.d03"): (
        "mismo objeto: el desplazamiento; las notas mencionan también el chip, que es otro objeto"
    ),
    ("ES2016.p002", "ES2016c.d03"): "mismo objeto: el chip-on-print",
    ("ES2008.p013", "ES2008c.d07"): "d.d05 nombra el botón de encendido",
    ("ES2008.p009", "ES2008c.d07"): (
        "DÉBIL: sin ancla textual; misma hija que en p013 por consistencia"
    ),
}
REASSIGNED["criterio"] = [
    CRITERION[(p, a)] for p, a in zip(REASSIGNED["pair_id"], REASSIGNED["antes"], strict=True)
]
REASSIGNED["cambio"] = REASSIGNED["antes"] + " -> " + REASSIGNED["después"]


def order(p):
    return (p.split(".")[0], int(p.split(".p")[1]))


REASSIGNED = REASSIGNED.sort_values("pair_id", key=lambda c: c.map(order)).reset_index(drop=True)
REASSIGNED[["pair_id", "serie", "cambio", "criterio"]]

,pair_id,serie,cambio,criterio
0,ES2008.p009,ES2008,ES2008c.d07 -> ES2008c.d07-1,DÉBIL: sin ancla textual; misma hija que en p0...
1,ES2008.p013,ES2008,ES2008c.d07 -> ES2008c.d07-1,d.d05 nombra el botón de encendido
2,ES2014.p010,ES2014,ES2014b.d04 -> ES2014b.d04-3,la nota de gm nombra b.d04-3
3,ES2014.p011,ES2014,ES2014b.d04 -> ES2014b.d04-9,d.d07 nombra el botón de confirmación; la nota...
4,ES2014.p012,ES2014,ES2014c.d07 -> ES2014c.d07-1,la nota de gm nombra c.d07-1
5,ES2014.p012,ES2014,ES2014d.d01 -> ES2014d.d01-1,la nota de gm nombra d.d01-1
6,ES2014.p014,ES2014,ES2014c.d02 -> ES2014c.d02-2,la nota de gm nombra c.d02-2
7,ES2016.p001,ES2016,ES2016c.d03 -> ES2016c.d03-1,mismo objeto: el desplazamiento; las notas men...
8,ES2016.p002,ES2016,ES2016c.d03 -> ES2016c.d03-2,mismo objeto: el chip-on-print
9,IS1008.p007,IS1008,IS1008b.d02 -> IS1008b.d02-2,la nota de gm nombra b.d02-2


# 5. Comprobaciones

Cada hija elegida existe en la referencia, es hija del padre reemplazado, y la relación reasignada
figura entre las aceptadas de su serie con la hija como extremo.

In [6]:
checks = []
for _, r in REASSIGNED.iterrows():
    kids = COMPUESTAS[r["serie"]].get(r["antes"], [])
    accepted = {(rel.earlier_id, rel.later_id) for rel in load_gold_relations(r["serie"])}
    appears = any(r["después"] in pair for pair in accepted)
    checks.append(
        {
            "pair_id": r["pair_id"],
            "hijas del padre": len(kids),
            "la hija existe": r["después"] in kids,
            "figura en las aceptadas": appears,
        }
    )
CHECKS = pd.DataFrame(checks)
assert CHECKS["la hija existe"].all() and CHECKS["figura en las aceptadas"].all()
print("Las", len(CHECKS), "reasignaciones apuntan a una hija existente y figuran como aceptadas.")
CHECKS

Las 11 reasignaciones apuntan a una hija existente y figuran como aceptadas.


,pair_id,hijas del padre,la hija existe,figura en las aceptadas
0,ES2008.p009,4,True,True
1,ES2008.p013,4,True,True
2,ES2014.p010,9,True,True
3,ES2014.p011,9,True,True
4,ES2014.p012,2,True,True
5,ES2014.p012,2,True,True
6,ES2014.p014,2,True,True
7,ES2016.p001,2,True,True
8,ES2016.p002,2,True,True
9,IS1008.p007,5,True,True


# 6. Lectura y límites

- Tras la corrección, ninguna de las relaciones aceptadas de las once series tiene como extremo
  una decisión `compuesta` (tabla de la sección 3). El mismo requisito queda fijado por
  `tests/test_gold_relations_endpoints.py`, que falla si una relación futura vuelve a apuntar a un
  padre.
- El conteo de relaciones no cambia: se reasigna un extremo, no se agregan ni se quitan pares
  (una relación, un enlace). Por eso el mínimo de E3 del ADR 0021 no se altera.
- Quedan fuera de esta decisión las demás hijas que las notas de la anotación también respaldan:
  incluirlas elevaría el número de enlaces de la referencia y contradiría el ADR 0021, donde las
  hijas cuentan como una sola relación.
- La elección de la hija es una decisión de contenido de `jss`; ES2008.p009 es la única sin ancla
  textual y se declara débil.
- Este notebook no mide nada del Experimento A ni del B: solo comprueba la estructura de la
  referencia de relaciones.